# 11785 HW3P2: Automatic Speech Recognition

Welcome to HW3P2. In this homework, you will be using the same data from HW1 but will be incorporating sequence models. We recommend you get familaried with sequential data and the working of RNNs, LSTMs and GRUs to have a smooth learning in this part of the homework.

Disclaimer: This starter notebook will not be as elaborate as that of HW1P2 or HW2P2. You will need to do most of the implementation in this notebook because, it is expected after 2 HWs, you will be in a position to write a notebook from scratch. You are welcomed to reuse the code from the previous starter notebooks but may also need to make appropriate changes for this homework. <br>
We have also given you 3 log files for the Very Low Cutoff (Levenshtein Distance = 30) so that you can observe how loss decreases.

Common errors which you may face


*   Shape errors: Half of the errors from this homework will account to this category. Try printing the shapes between intermediate steps to debug
*   CUDA out of Memory: When your architecture has a lot of parameters, this can happen. Golden keys for this is, (1) Reducing batch_size (2) Call *torch.cuda.empty_cache* often, even inside your training loop, (3) Call *gc.collect* if it helps and (4) Restart run time if nothing works







# Prelimilaries

You will need to install packages for decoding and calculating the Levenshtein distance

In [ ]:
!gdown --fuzzy https://drive.google.com/u/0/uc?id=1-sMVUu075veX9cx-LDYB9cn8GhyZvQ2M&export=download


# Libraries

In [ ]:
!pip install python-Levenshtein
!git clone --recursive https://github.com/parlance/ctcdecode.git
!pip install wget
%cd ctcdecode
!pip install .
%cd ..

!pip install torchsummaryX # We also install a summary package to check our model's forward before training

     |████████████████████████████████| 50 kB 2.9 MB/s 
  Created wheel for python-Levenshtein: filename=python_Levenshtein-0.12.2-cp37-cp37m-linux_x86_64.whl size=149873 sha256=608153c587381077eec97823c44b8858ff09ce2e8412fdcbd2188ec5e522b37e
  Stored in directory: /root/.cache/pip/wheels/05/5f/ca/7c4367734892581bb5ff896f15027a932c551080b2abd3e00d
Successfully built python-Levenshtein
Cloning into 'ctcdecode'...
remote: Enumerating objects: 1102, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 1102 (delta 16), reused 32 (delta 14), pack-reused 1063
Receiving objects: 100% (1102/1102), 782.27 KiB | 13.72 MiB/s, done.
Resolving deltas: 100% (529/529), done.
Submodule 'third_party/ThreadPool' (https://github.com/progschj/ThreadPool.git) registered for path 'third_party/ThreadPool'
Submodule 'third_party/kenlm' (https://github.com/kpu/kenlm.git) registered for path 'third_party/kenlm'
Cloning into '/content/ctcdecode/third_

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchsummaryX import summary

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

from sklearn.metrics import accuracy_score
import gc
import zipfile
import pandas as pd
from tqdm import tqdm
import os
import datetime

# imports for decoding and distance calculation
import ctcdecode
import Levenshtein
from ctcdecode import CTCBeamDecoder

import torch.optim as optim
import statistics
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Kaggle (TODO)

You need to set up your Kaggle and download the data

In [ ]:
import json

TOKEN = {"username":"******","key":"**********"}

!pip install kaggle==1.5.12
!mkdir -p .kaggle
!mkdir -p /content & mkdir -p /content/.kaggle & mkdir -p /root/.kaggle/

with open('/content/.kaggle/kaggle.json', 'w') as file:
    json.dump(TOKEN, file)

# ! pip install --upgrade --force-reinstall --no-deps kaggle
!ls "/content/.kaggle"
!chmod 600 /content/.kaggle/kaggle.json
!cp /content/.kaggle/kaggle.json /root/.kaggle/

!kaggle config set -n path -v /content

kaggle.json
- path is now set to: /content


In [ ]:
!kaggle competitions download -c 11-785-s22-hw4p2

 99% 1.82G/1.84G [00:10<00:00, 182MB/s]
100% 1.84G/1.84G [00:10<00:00, 189MB/s]


In [ ]:
!unzip -q /content/competitions/11-785-s22-hw4p2/11-785-s22-hw4p2.zip -d /content


# Dataset and dataloading (TODO)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import Levenshtein as lev
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.rnn as rnn_utils
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.nn.utils as utils
import seaborn as sns
import matplotlib.pyplot as plt
import time
import random
import datetime
from torch.utils import data
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

from tqdm import tqdm

cuda = torch.cuda.is_available()

print(cuda, sys.version)

device = torch.device("cuda" if cuda else "cpu")
num_workers = 4 if cuda else 0
print("Cuda = "+str(cuda)+" with num_workers = "+str(num_workers))
np.random.seed(11785)
torch.manual_seed(11785)

# The labels of the dataset contain letters in LETTER_LIST.
# You should use this to convert the letters to the corresponding indices
# and train your model with numerical labels.
LETTER_LIST = ['<sos>', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', \
         'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', "'", ' ', '<eos>']

True 3.7.13 (default, Apr 24 2022, 01:04:09) 
[GCC 7.5.0]
Cuda = True with num_workers = 4


In [ ]:
# This cell is where your actual TODOs start
# You will need to implement the Dataset class by your own. You may also implement it similar to HW1P2 (dont require context)
# The steps for implementation given below are how we have implemented it.
# However, you are welcomed to do it your own way if it is more comfortable or efficient.

class LibriSamples(torch.utils.data.Dataset):

    def __init__(self, data_path, partition= "train"): # You can use partition to specify train or dev

        self.X_dir = data_path + "/" + partition + "/mfcc/"       # TODO: get mfcc directory path
        self.Y_dir = data_path + "/" + partition + "/transcript/" # TODO: get transcript path

        # print("got mfcc directory path which is ", type(self.X_dir))
        # print("got transcipt path which is ", type(self.Y_dir))

        self.X_files = os.listdir(self.X_dir) # TODO: list files in the mfcc directory
        self.Y_files = os.listdir(self.Y_dir) # TODO: list files in the transcript directory

        # TODO: store PHONEMES from phonemes.py inside the class. phonemes.py will be downloaded from kaggle.
        # You may wish to store PHONEMES as a class attribute or a global variable as well.
        self.LETTER_LIST = LETTER_LIST #NotImplemented

        assert(len(self.X_files) == len(self.Y_files))

        # self.length = len(self.X_files) #added by me
        # pass

    def __len__(self):
        return len(self.X_files)

    def __getitem__(self, ind):
        # print("1. Getting Items... lol")
        # print("Trying to get ", self.X_files[ind])

        X = np.load(self.X_dir + self.X_files[ind]) # TODO: Load the mfcc npy file at the specified index ind in the directory
        Y = np.load(self.Y_dir + self.Y_files[ind]) # TODO: Load the corresponding transcripts
        X = (X - X.mean(axis=0))/X.std(axis=0)

        # print("Still working ")
        yyy = [self.LETTER_LIST.index(yy) for yy in Y[1:-1]]
        # print("Done working ")

        # Remember, the transcripts are a sequence of phonemes. Eg. np.array(['<sos>', 'B', 'IH', 'K', 'SH', 'AA', '<eos>'])
        # You need to convert these into a sequence of Long tensors
        # Tip: You may need to use self.PHONEMES
        # Remember, PHONEMES or PHONEME_MAP do not have '<sos>' or '<eos>' but the transcripts have them.
        # You need to remove '<sos>' and '<eos>' from the trancripts.
        # Inefficient way is to use a for loop for this. Efficient way is to think that '<sos>' occurs at the start and '<eos>' occurs at the end.

        Yy = torch.tensor(yyy, dtype=torch.long) # TODO: Convert sequence of  phonemes into sequence of Long tensors

        # print("1. Getting Items... Passed lol")

        return X, Yy

    def collate_fn(self, batch):
        # print("1. Test Collate executing 101")

        batch_x = [torch.tensor(x) for x,y in batch]
        batch_y = [torch.tensor(y) for x,y in batch]

        # TODO: pad the sequence with pad_sequence (already imported)
        batch_x_pad = pad_sequence(batch_x, batch_first=True, padding_value = 0)
        lengths_x = [len(x) for x, y in batch]
        # lengths_x = len([x for x, y in batch_x]) # TODO: Get original lengths of the sequence before padding

        batch_y_pad = pad_sequence(batch_y, batch_first=True, padding_value = 0)# TODO: pad the sequence with pad_sequence (already imported)
        lengths_y = [len(y) for x, y in batch] # TODO: Get original lengths of the sequence before padding

        # print("1. Test Collate executed 101")

        return batch_x_pad, batch_y_pad, torch.tensor(lengths_x), torch.tensor(lengths_y)


# You can either try to combine test data in the previous class or write a new Dataset class for test data
class LibriSamplesTest(torch.utils.data.Dataset):

    def __init__(self, data_path, test_order): # test_order is the csv similar to what you used in hw1\

        path_test = data_path + "/test/" + test_order
        test_order_list = pd.read_csv(path_test)  # TODO: open test_order.csv as a list
        self.X = test_order_list["file"] # TODO: Load the npy files from test_order.csv and append into a list
        # You can load the files here or save the paths here and load inside __getitem__ like the previous class

        self.X_dir = data_path + "/test/mfcc/"

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ind):
        # TODOs: Need to return only X because this is the test dataset
        # return NotImplemented
        X = np.load(self.X_dir + self.X[ind])
        X = (X - X.mean(axis=0))/X.std(axis=0)
        return X


    def collate_fn(self, batch):
        # print("Test Collate executing 101")
        batch_x = [torch.tensor(x) for x in batch]
        # TODO: pad the sequence with pad_sequence (already imported)
        batch_x_pad = pad_sequence(batch_x, batch_first=True, padding_value = 0)
        # TODO: Get original lengths of the sequence before padding
        lengths_x = [len(x) for x in batch]
        # print("Test Collate executed 101")

        return batch_x_pad, torch.tensor(lengths_x)

In [ ]:
batch_size = 96  #128

root = "/content/hw4p2_student_data/hw4p2_student_data" # TODO: Where your hw3p2_student_data folder is

train_data  = LibriSamples(root, "train")
val_data    = LibriSamples(root, "dev")
test_data   = LibriSamplesTest(root, 'test_order.csv')

# TODO: Define the train loader. Remember to pass in a parameter (function) for the collate_fn argument
# train_loader  = torch.utils.data.DataLoader(train_data, batch_size=batch_size)
# train_loader  = torch.utils.data.DataLoader(train_data, batch_size, collate_fn=train_data.collate_fn)
train_loader  = torch.utils.data.DataLoader(train_data, batch_size, shuffle= True, collate_fn=train_data.collate_fn)

# TODO: Define the val loader. Remember to pass in a parameter (function) for the collate_fn argument
val_loader    = torch.utils.data.DataLoader(val_data, batch_size, shuffle= False, collate_fn=val_data.collate_fn)

# TODO: Define the test loader. Remember to pass in a parameter (function) for the collate_fn argument
test_loader   = torch.utils.data.DataLoader(test_data, batch_size, shuffle= False, collate_fn=test_data.collate_fn)

print("Batch size: ", batch_size)
print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Val dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Batch size:  96
Train dataset samples = 28539, batches = 298
Val dataset samples = 2703, batches = 29
Test dataset samples = 2620, batches = 28


In [ ]:
# Optional

#Test code for checking shapes and return arguments of the train and val loaders
for data in val_loader:
  x, y, lx, ly = data # if you face an error saying "Cannot unpack", then you are not passing the collate_fn argument
    # print(x.shape, y.shape, lx.shape, ly.shape)

# Model Configuration (TODO)

In [ ]:
class Network(nn.Module):

    def __init__(self, input_size, out_channels, num_layers, num_classes, hidden_size=256): # You can add any extra arguments as you wish

        super(Network, self).__init__()


        self.conv_stack1 = nn.Sequential(
            nn.Conv1d(input_size, out_channels, kernel_size = 5, stride=1),#, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.MaxPool1d(kernel_size= 2, stride= 2),
            nn.Dropout(0.5)
        )

        self.conv_stack2 = nn.Sequential(
            nn.Conv1d(out_channels, 512, kernel_size = 5, stride=1), #, padding=1),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.MaxPool1d(kernel_size= 2, stride= 2),
            nn.Dropout(0.5)
        )


        # try 512 next
        self.lstm = nn.LSTM(512, hidden_size, num_layers,  batch_first=True, dropout = 0.5, bidirectional=True)# TODO: # Create a single layer, uni-directional LSTM with hidden_size = 256
        # Use nn.LSTM() Make sure that you give in the proper arguments as given in https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html


        self.classifications_stack = nn.Sequential(

            # TODO: Create a single classification layer using nn.Linear()
            nn.Linear(hidden_size*2, 2048),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(2048, 2048), # try 4096 next
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(2048, 2048), # try 4096 next
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(2048, 30),
        )

        self.log_softmax = nn.LogSoftmax(dim=2)

    # def forward(self, x): # TODO: You need to pass atleast 1 more parameter apart from self and x
    def forward(self, x, lengths_x): # TODO: You need to pass atleast 1 more parameter apart from self and x

        x = torch.transpose(x, 1, 2)
        out = self.conv_stack1(x)
        out = self.conv_stack2(out)
        lengths_x = torch.clamp(lengths_x, max=out.shape[2])
        out = torch.transpose(out, 1, 2)

        # print(out.shape)

        packed_input = torch.nn.utils.rnn.pack_padded_sequence(out, batch_first=True, lengths=lengths_x, enforce_sorted=False)

        out1, (out2, out3) = self.lstm(packed_input)# TODO: Pass packed input to self.lstm

        out, lengths  = pad_packed_sequence(out1, batch_first=True)# TODO: Need to 'unpack' the LSTM output using pad_packed_sequence

        out = self.classifications_stack(out)# TODO: Pass unpacked LSTM output to the classification layer

        out = self.log_softmax(out)# Optional: Do log softmax on the output. Which dimension?

        return out, lengths # TODO: Need to return 2 variables




model = Network(13, 256, 4,30).cuda()
print(model)
summary(model, x.to(device), lx) # x and lx are from the previous cell


Network(
  (conv_stack1): Sequential(
    (0): Conv1d(13, 256, kernel_size=(5,), stride=(1,))
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.5, inplace=False)
  )
  (conv_stack2): Sequential(
    (0): Conv1d(256, 512, kernel_size=(5,), stride=(1,))
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.5, inplace=False)
  )
  (lstm): LSTM(512, 256, num_layers=4, batch_first=True, dropout=0.5, bidirectional=True)
  (classifications_stack): Sequential(
    (0): Linear(in_features=512, out_features=2048, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=2048, out_features=2048, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inp

,Kernel Shape,Output Shape,Params,Mult-Adds
Layer,,,,
0_conv_stack1.Conv1d_0,"[13, 256, 5]","[15, 256, 1565]",16896.0,26041600.0
1_conv_stack1.BatchNorm1d_1,[256],"[15, 256, 1565]",512.0,256.0
2_conv_stack1.GELU_2,-,"[15, 256, 1565]",NaN,NaN
3_conv_stack1.MaxPool1d_3,-,"[15, 256, 782]",NaN,NaN
4_conv_stack1.Dropout_4,-,"[15, 256, 782]",NaN,NaN
5_conv_stack2.Conv1d_0,"[256, 512, 5]","[15, 512, 778]",655872.0,509870080.0
6_conv_stack2.BatchNorm1d_1,[512],"[15, 512, 778]",1024.0,512.0
7_conv_stack2.GELU_2,-,"[15, 512, 778]",NaN,NaN
8_conv_stack2.MaxPool1d_3,-,"[15, 512, 389]",NaN,NaN


# Training Configuration (TODO)

In [ ]:
# # Check out https://github.com/parlance/ctcdecode for the details on how to implement decoding
# # Do you need to give log_probs_input = True or False?

criterion = nn.CTCLoss().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

# TODO: Intialize the CTC beam decoder
decoder = ctcdecode.CTCBeamDecoder(LETTER_LIST, beam_width=50, num_processes=4, log_probs_input=True)

# Do you need to give log_probs_input = True or False?
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.8, min_lr=0.0004, patience=3, verbose=False)


In [ ]:
# this function calculates the Levenshtein distance

def calculate_levenshtein(h, y, lh, ly, decoder, LETTER_LIST):
    h = h.float().to(device)
    y = y.float().to(device)

    # h - ouput from the model. Probability distributions at each time step
    # y - target output sequence - sequence of Long tensors
    # lh, ly - Lengths of output and target
    # decoder - decoder object which was initialized in the previous cell
    # PHONEME_MAP - maps output to a character to find the Levenshtein distance
    # TODO: You may need to transpose or permute h based on how you passed it to the criterion

    h = torch.transpose(h, 0, 1)

    beam_results, beam_scores, timesteps, out_lens = decoder.decode(h, seq_lens=lh)

    # Input to the decode method will be h and its lengths lh
    # You need to pass lh for the 'seq_lens' parameter. This is not explicitly mentioned in the git repo of ctcdecode.

    batch_size = beam_results.shape[0] # TODO

    dist = 0

    # Loop through each element in the batch
    for i in range(batch_size):

        h_sliced = beam_results[i][0][:out_lens[i][0]]

        h_chars = [LETTER_LIST[int(x)] for x in h_sliced]

        # TODO: MAP the sequence of numbers to its corresponding characters with PHONEME_MAP and merge everything as a single string
        h_string = ''.join(h_chars)

        yy = y[i]

        lyy = ly[i]

        yyyyy = yy[:lyy]

        y_chars = [LETTER_LIST[int(c)] for c in yyyyy]

        y_string = ''.join(y_chars)

        dist += Levenshtein.distance(h_string, y_string)

    dist/=batch_size

    return dist

In [ ]:
def validate_model(model):
    model.eval()
    validation_bar = tqdm(total=len(val_loader), dynamic_ncols=True, position=0, leave=False, desc='Validation')
    dists = []
    loss = 0
    count = 0
    for i, _data in enumerate(val_loader):
        data, target, input_lengths, target_lengths = _data
        data = data.float().to(device)
        target = target.float().to(device)
        input_lengths = torch.tensor(input_lengths, dtype=torch.long)
        target_lengths = torch.tensor(target_lengths, dtype=torch.long)
        with torch.no_grad():
            output, lenghts= model(data, input_lengths)
        output = torch.transpose(output, 0, 1)

        loss += criterion(output, target, input_lengths=lenghts, target_lengths=target_lengths)
        dist = calculate_levenshtein(output, target, input_lengths, target_lengths, decoder, LETTER_LIST)
        dists.append(dist)
        validation_bar.set_description(f"Levenshtein distance: {dist}")
        validation_bar.update()
        count += 1

    val_loss = loss/count
    validation_bar.close()
    return statistics.mean(dists), val_loss

In [ ]:
epochs = 15

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd drive/MyDrive/deep_learning/hw4p2/

In [ ]:
checkpoint = torch.load('/content/drive/MyDrive/model_epoch_15.path')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']

# model.eval()
# # - or -
# model.train()

In [ ]:
# model.state_dict

In [ ]:
# Optional but recommended
torch.cuda.empty_cache()

for epoch in range(15):
    model.train()
    batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')
    dist  = 0
    tloss = 0
    count = 0
    for i, _data in enumerate(train_loader):

        data, target, input_lengths, target_lengths = _data
        data = data.float().to(device)
        target = target.float().to(device)

        input_lengths = torch.tensor(input_lengths, dtype=torch.long)
        target_lengths = torch.tensor(target_lengths, dtype=torch.long)
        optimizer.zero_grad()

        output, lenghts= model(data, input_lengths)
        output = torch.transpose(output, 0, 1)

        loss = criterion(output, target, input_lengths=lenghts, target_lengths=target_lengths)
        tloss += loss
        loss.backward()
        optimizer.step()
        count += 1

        state = {
            'epoch':epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
            'optimizer_state_dict': optimizer.state_dict(),
            }

        torch.save(state, f"/content/drive/MyDrive/model_epoch_{epoch+1}.path")

        batch_bar.update()

        batch_bar.set_description('Train Epoch: {}/{} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch+1,epochs, i * len(_data), len(train_loader.dataset),
                100. * i / len(train_loader), loss.item()))


    trainloss = tloss/count
    dists, val_loss = validate_model(model)
    scheduler.step(val_loss)


    print(f"Epoch: {epoch+1}/{epochs}: Train Loss {trainloss}, Val Loss {val_loss}, Levenshtein dist {dists}", )
    batch_bar.close()

Epoch: 1/15: Train Loss 0.43649575114250183, Val Loss 0.41234341263771057, Levenshtein dist 9.786350574712644


Epoch: 2/15: Train Loss 0.4744964838027954, Val Loss 0.4583565592765808, Levenshtein dist 10.838074712643678


Epoch: 3/15: Train Loss 0.46124687790870667, Val Loss 0.4235818386077881, Levenshtein dist 10.027155172413794


Epoch: 4/15: Train Loss 0.44011190533638, Val Loss 0.41922953724861145, Levenshtein dist 10.13441091954023


Epoch: 5/15: Train Loss 0.4266098141670227, Val Loss 0.41374117136001587, Levenshtein dist 9.641091954022988


...

Epoch: 12/15: Train Loss 0.4362802803516388, Val Loss 0.4045596718788147, Levenshtein dist 9.74044540229885


Epoch: 13/15: Train Loss 0.41643065214157104, Val Loss 0.3968455493450165, Levenshtein dist 9.471048850574713


Epoch: 14/15: Train Loss 0.40387800335884094, Val Loss 0.39136412739753723, Levenshtein dist 9.370186781609195


Train Epoch: 15/15 [116/28539 (10%)]	Loss: 0.427216:  10%|█         | 30/298 [01:04<09:37,  2.16s/it]

KeyboardInterrupt: ignored

In [ ]:
model.eval()

batch_bar = tqdm(total=len(val_loader), dynamic_ncols=True, position=0, leave=False, desc='Val')
dist = 0

for i, _data in enumerate(val_loader):
    data, target, input_lengths, target_lengths = _data
    data = data.float().to(device)
    target = target.float().to(device)

    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)

    with torch.no_grad():
        output, lenghts= model(data, input_lengths)

    output = torch.transpose(output, 0, 1)

    dist = calculate_levenshtein(output, target, input_lengths, target_lengths, decoder, LETTER_LIST)
    print(f"validation distance found is {dist}")

In [ ]:
# package and submit
predictions = []
model.eval()
batch_bar = tqdm(total=len(test_loader), dynamic_ncols=True, position=0, leave=False, desc='Val')
dist = 0
for i, _data in enumerate(test_loader):
    data, input_lengths = _data
    data = data.float().to(device)

    input_lengths = torch.tensor(input_lengths, dtype=torch.long)

    with torch.no_grad():
        output, lenghts= model(data, input_lengths)

    output = torch.transpose(output, 0, 1)

    preds = calculate_levenshtein_test(output, lenghts, decoder, LETTER_LIST)

    predictions.extend(preds)

ids = np.arange(len(predictions))
dat = pd.DataFrame({"id": ids, "predictions":predictions})
dat.to_csv("slack_submission.csv", index=False)

In [ ]:
!kaggle competitions submit -c 11-785-s22-hw4p2-slack -f slack_submission.csv -m "Message"


#Initial codes

In [ ]:
# Optional but recommended

for i, data in enumerate(train_loader, 0):

    # Write a test code do perform a single forward pass and also compute the Levenshtein distance
    # Make sure that you are able to get this right before going on to the actual training
    # You may encounter a lot of shape errors
    # Printing out the shapes will help in debugging
    # Keep in mind that the Loss which you will use requires the input to be in a different format and the decoder expects it in a different format
    # Make sure to read the corresponding docs about it
    pass

    break # one iteration is enough

In [ ]:
torch.cuda.empty_cache() # Use this often

# TODO: Write the model evaluation function if you want to validate after every epoch

# You are free to write your own code for model evaluation or you can use the code from previous homeworks' starter notebooks
# However, you will have to make modifications because of the following.
# (1) The dataloader returns 4 items unlike 2 for hw2p2
# (2) The model forward returns 2 outputs
# (3) The loss may require transpose or permuting

# Note that when you give a higher beam width, decoding will take a longer time to get executed
# Therefore, it is recommended that you calculate only the val dataset's Levenshtein distance (train not recommended) with a small beam width
# When you are evaluating on your test set, you may have a higher beam width

In [ ]:
torch.cuda.empty_cache()

# TODO: Write the model training code

# You are free to write your own code for training or you can use the code from previous homeworks' starter notebooks
# However, you will have to make modifications because of the following.
# (1) The dataloader returns 4 items unlike 2 for hw2p2
# (2) The model forward returns 2 outputs
# (3) The loss may require transpose or permuting

# Tip: Implement mixed precision training

# Submit to kaggle (TODO)

In [ ]:
# TODO: Write your model evaluation code for the test dataset
# You can write your own code or use from the previous homewoks' stater notebooks
# You can't calculate loss here. Why?

In [ ]:
# TODO: Generate the csv file